# Adobe (ADBE) Stock Data — Data Analyst Project

## Project Objective
Analyze Adobe (ADBE) historical stock-market data to understand:
- Data quality and structure
- Price and trading-volume trends
- Daily returns and volatility
- Moving averages
- High/low price behavior
- Trading-volume patterns
- Business insights that could support investment monitoring

> **Note:** This is an analytical/educational project, not financial advice.


## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the uploaded dataset
df = pd.read_csv("ADBE.csv")

# Clean column names
df.columns = df.columns.str.strip()

# Convert Date if available
if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.sort_values("Date").reset_index(drop=True)

df.head()


## 2. Dataset Overview

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())


## 3. Data Quality Check

In [ ]:
quality = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2),
    "Unique Values": df.nunique()
})
display(quality)

print("Duplicate rows:", df.duplicated().sum())


## 4. Descriptive Statistics

In [ ]:
display(df.describe(include="all").T)


## 5. Identify Core Stock Columns

In [ ]:
# Common OHLCV columns
available = {c.lower(): c for c in df.columns}

def find_col(name):
    return available.get(name.lower())

date_col = find_col("Date")
open_col = find_col("Open")
high_col = find_col("High")
low_col = find_col("Low")
close_col = find_col("Close")
volume_col = find_col("Volume")

print({
    "Date": date_col,
    "Open": open_col,
    "High": high_col,
    "Low": low_col,
    "Close": close_col,
    "Volume": volume_col
})


## 6. Closing Price Trend

In [ ]:
if date_col and close_col:
    plt.figure(figsize=(12, 5))
    plt.plot(df[date_col], df[close_col])
    plt.title("ADBE Closing Price Over Time")
    plt.xlabel("Date")
    plt.ylabel("Closing Price")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 7. Trading Volume Trend

In [ ]:
if date_col and volume_col:
    plt.figure(figsize=(12, 5))
    plt.plot(df[date_col], df[volume_col])
    plt.title("ADBE Trading Volume Over Time")
    plt.xlabel("Date")
    plt.ylabel("Volume")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 8. Daily Returns Analysis

Daily return measures the percentage change in closing price from one trading day to the next.

**Formula:**

Daily Return = (Today's Close / Yesterday's Close) - 1


In [ ]:
if close_col:
    df["Daily_Return"] = df[close_col].pct_change()
    display(df[[date_col, close_col, "Daily_Return"]].head(10) if date_col else df[[close_col, "Daily_Return"]].head(10))


## 9. Return Distribution

In [ ]:
if "Daily_Return" in df.columns:
    plt.figure(figsize=(10, 5))
    plt.hist(df["Daily_Return"].dropna(), bins=50)
    plt.title("Distribution of ADBE Daily Returns")
    plt.xlabel("Daily Return")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

    print("Average daily return:", round(df["Daily_Return"].mean() * 100, 4), "%")
    print("Daily volatility:", round(df["Daily_Return"].std() * 100, 4), "%")


## 10. Moving Averages

Moving averages smooth short-term fluctuations and help identify longer-term price trends.

We calculate:
- 20-day moving average
- 50-day moving average
- 200-day moving average (when enough observations exist)


In [ ]:
if close_col:
    df["MA_20"] = df[close_col].rolling(20).mean()
    df["MA_50"] = df[close_col].rolling(50).mean()
    df["MA_200"] = df[close_col].rolling(200).mean()

    plt.figure(figsize=(13, 6))
    plt.plot(df[date_col], df[close_col], label="Close")
    plt.plot(df[date_col], df["MA_20"], label="20-Day MA")
    plt.plot(df[date_col], df["MA_50"], label="50-Day MA")
    if df["MA_200"].notna().any():
        plt.plot(df[date_col], df["MA_200"], label="200-Day MA")
    plt.title("ADBE Price and Moving Averages")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 11. Volatility Analysis

A rolling standard deviation of daily returns helps identify periods when the stock experienced unusually high price movement.


In [ ]:
if "Daily_Return" in df.columns:
    df["Rolling_Volatility_20D"] = df["Daily_Return"].rolling(20).std()

    plt.figure(figsize=(12, 5))
    plt.plot(df[date_col], df["Rolling_Volatility_20D"])
    plt.title("20-Day Rolling Volatility")
    plt.xlabel("Date")
    plt.ylabel("Volatility")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 12. Daily Price Range

Price range = High - Low

This can be used as a simple measure of the amount of intraday movement.


In [ ]:
if high_col and low_col:
    df["Daily_Range"] = df[high_col] - df[low_col]

    plt.figure(figsize=(12, 5))
    plt.plot(df[date_col], df["Daily_Range"])
    plt.title("ADBE Daily High-Low Price Range")
    plt.xlabel("Date")
    plt.ylabel("Price Range")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 13. Best and Worst Trading Days

In [ ]:
if "Daily_Return" in df.columns:
    best_days = df.nlargest(10, "Daily_Return")
    worst_days = df.nsmallest(10, "Daily_Return")

    print("Top 10 Positive Return Days")
    display(best_days[[date_col, close_col, "Daily_Return"]])

    print("Top 10 Negative Return Days")
    display(worst_days[[date_col, close_col, "Daily_Return"]])


## 14. Highest and Lowest Closing Prices

In [ ]:
if close_col:
    max_idx = df[close_col].idxmax()
    min_idx = df[close_col].idxmin()

    print("Highest Closing Price:")
    display(df.loc[[max_idx], [date_col, close_col]])

    print("Lowest Closing Price:")
    display(df.loc[[min_idx], [date_col, close_col]])


## 15. Monthly Performance Analysis

In [ ]:
if date_col and "Daily_Return" in df.columns:
    monthly = (
        df.set_index(date_col)["Daily_Return"]
          .resample("ME")
          .apply(lambda x: (1 + x).prod() - 1)
          .dropna()
    )

    monthly_df = monthly.to_frame("Monthly_Return")
    monthly_df["Monthly_Return_%"] = monthly_df["Monthly_Return"] * 100

    display(monthly_df.tail(20))

    plt.figure(figsize=(13, 5))
    plt.bar(monthly_df.index, monthly_df["Monthly_Return_%"])
    plt.title("ADBE Monthly Returns")
    plt.xlabel("Month")
    plt.ylabel("Return (%)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 16. Yearly Performance

In [ ]:
if date_col and "Daily_Return" in df.columns:
    yearly = (
        df.set_index(date_col)["Daily_Return"]
          .resample("YE")
          .apply(lambda x: (1 + x).prod() - 1)
          .dropna()
    )
    yearly_df = yearly.to_frame("Annual_Return")
    yearly_df["Annual_Return_%"] = yearly_df["Annual_Return"] * 100
    display(yearly_df)


## 17. Drawdown Analysis

Drawdown measures how far the stock falls from its previous peak.

This is useful for understanding downside periods and recovery behavior.


In [ ]:
if close_col:
    running_max = df[close_col].cummax()
    df["Drawdown"] = df[close_col] / running_max - 1

    plt.figure(figsize=(12, 5))
    plt.plot(df[date_col], df["Drawdown"] * 100)
    plt.title("ADBE Drawdown Over Time")
    plt.xlabel("Date")
    plt.ylabel("Drawdown (%)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    print("Maximum drawdown:", round(df["Drawdown"].min() * 100, 2), "%")


## 18. Volume vs. Daily Return

A useful exploratory question is whether unusually high trading volume occurs alongside large price movements.


In [ ]:
if volume_col and "Daily_Return" in df.columns:
    plt.figure(figsize=(10, 6))
    plt.scatter(df[volume_col], df["Daily_Return"], alpha=0.5)
    plt.title("Trading Volume vs Daily Return")
    plt.xlabel("Volume")
    plt.ylabel("Daily Return")
    plt.tight_layout()
    plt.show()

    print("Correlation between volume and daily return:",
          round(df[volume_col].corr(df["Daily_Return"]), 4))


## 19. Key KPI Summary

In [ ]:
kpis = {}

if close_col:
    kpis["Start Closing Price"] = df[close_col].iloc[0]
    kpis["Latest Closing Price"] = df[close_col].iloc[-1]
    kpis["Highest Closing Price"] = df[close_col].max()
    kpis["Lowest Closing Price"] = df[close_col].min()

if "Daily_Return" in df.columns:
    kpis["Average Daily Return %"] = df["Daily_Return"].mean() * 100
    kpis["Daily Volatility %"] = df["Daily_Return"].std() * 100
    kpis["Positive Return Days"] = (df["Daily_Return"] > 0).sum()
    kpis["Negative Return Days"] = (df["Daily_Return"] < 0).sum()

if "Drawdown" in df.columns:
    kpis["Maximum Drawdown %"] = df["Drawdown"].min() * 100

kpi_df = pd.DataFrame(kpis.items(), columns=["KPI", "Value"])
display(kpi_df)


## 20. Analyst Insights

Run the following cell to automatically generate a concise summary from the dataset.


In [ ]:
print("DATA ANALYST SUMMARY")
print("=" * 60)

if date_col:
    print(f"Analysis period: {df[date_col].min().date()} to {df[date_col].max().date()}")

if close_col:
    start_price = df[close_col].iloc[0]
    end_price = df[close_col].iloc[-1]
    total_change = (end_price / start_price - 1) * 100
    print(f"Overall price change: {total_change:.2f}%")

if "Daily_Return" in df.columns:
    print(f"Average daily return: {df['Daily_Return'].mean()*100:.4f}%")
    print(f"Daily volatility: {df['Daily_Return'].std()*100:.4f}%")
    print(f"Positive trading days: {(df['Daily_Return'] > 0).sum()}")
    print(f"Negative trading days: {(df['Daily_Return'] < 0).sum()}")

if "Drawdown" in df.columns:
    print(f"Maximum drawdown: {df['Drawdown'].min()*100:.2f}%")

print("\nInterpretation:")
print("- Use the price trend and moving averages to identify broad trend periods.")
print("- Use daily returns and rolling volatility to identify riskier periods.")
print("- Use drawdown to understand the severity of declines from previous peaks.")
print("- Compare volume spikes with large returns to identify unusual trading activity.")


## 21. Business Questions for an Interview

1. What was the overall price trend during the analysis period?
2. Which periods had the highest volatility?
3. What was the maximum drawdown?
4. How many trading days produced positive vs. negative returns?
5. Did high-volume days coincide with large price movements?
6. How could a dashboard help an analyst monitor ADBE?
7. Which KPIs would you show to a business stakeholder?
8. What additional external data would improve this analysis?

### Recommended Dashboard KPIs
- Latest Close
- Total Return
- Average Daily Return
- Volatility
- Maximum Drawdown
- Trading Volume
- 20/50/200-Day Moving Average
- Best Trading Day
- Worst Trading Day


## 22. Conclusion

This project demonstrates a complete **Data Analyst workflow**:

**Business Objective → Data Loading → Data Cleaning → EDA → KPI Analysis → Trend Analysis → Risk Analysis → Insights**

The project can be extended into a **Power BI dashboard** with:
- KPI cards
- Interactive date filters
- Closing-price trend
- Volume chart
- Monthly returns
- Drawdown
- Moving-average trend
- Volatility analysis
